# Step 4A.6 — перевірка середовища перед B3 (нічого НЕ завантажуємо)
Перевіряє: чи встановлені `torch`/`transformers`, чи є локально закешовані моделі
(SecBERT чи будь-яка cybersecurity-domain transformer-модель), яка в них embedding
dimension, і чи працює inference на локально доступній моделі (якщо є).
Якщо нічого не знайдено — ноутбук лише повідомляє про це, НЕ пропонує автоматично
завантажити щось нове.

Ядро: IDS Research.

In [1]:
import importlib.util
print("=" * 70)
print("1. Чи встановлені torch / transformers (без встановлення)")
print("=" * 70)

torch_spec = importlib.util.find_spec("torch")
transformers_spec = importlib.util.find_spec("transformers")

print(f"torch:        {'встановлено' if torch_spec else 'НЕ встановлено'}")
print(f"transformers: {'встановлено' if transformers_spec else 'НЕ встановлено'}")

if torch_spec:
    import torch
    print(f"  torch version: {torch.__version__}")
    print(f"  CUDA доступна: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
if transformers_spec:
    import transformers
    print(f"  transformers version: {transformers.__version__}")

1. Чи встановлені torch / transformers (без встановлення)
torch:        встановлено
transformers: НЕ встановлено
  torch version: 2.5.1+cu124
  CUDA доступна: False


/home/ivan_r_opirskyy/ids_research/lib/python3.11/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 803: system has unsupported display driver / cuda driver combination (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0


In [2]:
# ---------- 2. Пошук локально закешованих моделей (HuggingFace hub cache) ----------
from pathlib import Path

print("=" * 70)
print("2. Локальний Hugging Face cache")
print("=" * 70)

CACHE_CANDIDATES = [
    Path.home() / ".cache" / "huggingface" / "hub",
    Path("/srv/ids_research/.cache/huggingface/hub"),
    Path("/srv/ids_research/models"),
]

found_any = False
for cache_dir in CACHE_CANDIDATES:
    if cache_dir.exists():
        entries = sorted(p.name for p in cache_dir.iterdir())
        if entries:
            found_any = True
            print(f"\n{cache_dir}: {len(entries)} записів")
            for e in entries:
                print(f"  {e}")
        else:
            print(f"\n{cache_dir}: існує, але порожньо")
    else:
        print(f"\n{cache_dir}: не існує")

if not found_any:
    print("\n⚠ Жодної закешованої моделі не знайдено в перевірених місцях.")

2. Локальний Hugging Face cache

/home/ivan_r_opirskyy/.cache/huggingface/hub: не існує

/srv/ids_research/.cache/huggingface/hub: не існує

/srv/ids_research/models: не існує

⚠ Жодної закешованої моделі не знайдено в перевірених місцях.


In [ ]:
# ---------- 3. Прицільний пошук security-related моделей: HF-кеш + файлова система ----------
# Два джерела кандидатів:
#   (а) записи у HF cache (крок 2) — фільтруємо ЛИШЕ ті, чия назва містить хоч одне
#       з ключових слів нижче;
#   (б) файлова система за тими самими ключовими словами (на випадок моделі поза кешем).
import subprocess

print("=" * 70)
print("3. Ідентифікація SAME РЕЛЕВАНТНИХ (security/cybersecurity) моделей")
print("=" * 70)

SECURITY_KEYWORDS = ["secbert", "cybersec", "cyber", "security", "threat", "cve", "vuln", "cti"]

# (а) фільтруємо ВЖЕ знайдені у кроці 2 записи HF cache за ключовими словами
all_cache_entries = []
for cache_dir in CACHE_CANDIDATES:
    if cache_dir.exists():
        all_cache_entries.extend(cache_dir.iterdir())

relevant_cache_entries = [
    p for p in all_cache_entries
    if any(kw in p.name.lower() for kw in SECURITY_KEYWORDS)
]

print(f"Усього записів у HF cache: {len(all_cache_entries)}")
print(f"З них релевантних (за ключовими словами {SECURITY_KEYWORDS}): {len(relevant_cache_entries)}")
for p in relevant_cache_entries:
    print(f"  ✓ {p.name}")

irrelevant = [p.name for p in all_cache_entries if p not in relevant_cache_entries]
if irrelevant:
    print(f"\nІгноруються (не security-related, НЕ стануть кандидатами на B3 автоматично):")
    for name in irrelevant:
        print(f"  ✗ {name}")

# (б) пошук по файловій системі за ТИМИ САМИМИ ключовими словами (раніше шукались лише 4 з 8)
print("\nПошук по файловій системі за всіма ключовими словами (config.json/веса поза HF cache):")
find_args = ["find", "/home/ivan_r_opirskyy", "/srv/ids_research"]
for kw in SECURITY_KEYWORDS:
    find_args += ["-iname", f"*{kw}*", "-o"]
find_args = find_args[:-1]   # прибираємо зайвий трейлінговий "-o"

result = subprocess.run(find_args, capture_output=True, text=True, timeout=60)
print(result.stdout or "Нічого не знайдено за жодним із ключових слів.")
if result.stderr:
    print("stderr (permission denied тощо, можна ігнорувати):", result.stderr[:500])


In [ ]:
# ---------- 4. Sanity-check inference ЛИШЕ для РЕЛЕВАНТНИХ (security) кандидатів ----------
# КРИТИЧНО: беремо ТІЛЬКИ relevant_cache_entries з кроку 3, а не довільний перший запис
# усього кешу. Якщо relevant_cache_entries порожній — жодного кандидата на B3 немає,
# і ми НЕ підставляємо випадкову нерелевантну модель.
# local_files_only=True гарантує, що нічого не завантажується з інтернету.
#
# Це ЛИШЕ sanity-check ("чи модель взагалі відповідає і дає ембединг"), mean pooling тут —
# не остаточний вибір пулінгу для B3. Після вибору конкретного encoder pooling/max_length/
# batch size/normalize/обробка порожніх description фіксуються ОКРЕМО як B3 encoder spec.

sanity_results = []

if not relevant_cache_entries:
    print("Релевантних (security-related) моделей у HF cache не знайдено.")
    print("→ Sanity-check inference не виконується. Дивіться підсумок нижче.")
elif not (transformers_spec and torch_spec):
    print("torch/transformers не встановлені — sanity-check inference неможливий без встановлення.")
else:
    from transformers import AutoTokenizer, AutoModel
    import torch

    relevant_model_dirs = [p for p in relevant_cache_entries if p.name.startswith("models--")]
    if not relevant_model_dirs:
        print("Релевантні записи є, але жоден не має формату HF-репозиторію моделі (models--org--name).")
    for p in relevant_model_dirs:
        model_name = p.name.replace("models--", "").replace("--", "/")
        print(f"\n--- Кандидат: {model_name} ---")
        try:
            tok = AutoTokenizer.from_pretrained(model_name, local_files_only=True)
            model = AutoModel.from_pretrained(model_name, local_files_only=True)
            model.eval()

            sample_text = ("Remote code execution vulnerability in the authentication module "
                           "allows an attacker to execute arbitrary code.")
            inputs = tok(sample_text, return_tensors="pt", truncation=True, max_length=256)
            with torch.no_grad():
                output = model(**inputs)
            emb = output.last_hidden_state.mean(dim=1)   # ЛИШЕ для sanity-check, не остаточний pooling
            print(f"✅ Inference спрацював. Embedding dimension: {emb.shape[-1]}")
            sanity_results.append(dict(model_name=model_name, status="ok", embedding_dim=int(emb.shape[-1])))
        except Exception as e:
            print(f"❌ Не вдалося завантажити локально: {repr(e)[:300]}")
            print("   (local_files_only=True — нічого не завантажувалось з інтернету)")
            sanity_results.append(dict(model_name=model_name, status="failed", embedding_dim=None))


In [ ]:
# ---------- 5. ПІДСУМОК: чи є придатний security-domain encoder для B3 ----------
print("=" * 70)
print("ПІДСУМОК")
print("=" * 70)
print(f"torch встановлено:                    {bool(torch_spec)}")
print(f"transformers встановлено:              {bool(transformers_spec)}")
print(f"релевантних (security) моделей у кеші: {len(relevant_cache_entries)}")

working_candidates = [r for r in sanity_results if r["status"] == "ok"]

if working_candidates:
    print(f"\n✅ Знайдено {len(working_candidates)} робочих security-domain кандидат(ів) на B3 encoder:")
    for r in working_candidates:
        print(f"   {r['model_name']} (embedding_dim={r['embedding_dim']})")
    print("\n→ Наступний крок: обрати ОДИН із них як B3 encoder і зафіксувати окрему")
    print("  специфікацію (pooling, max_length, batch size, device, normalize, обробка")
    print("  порожніх description) — це вже НЕ environment check, а B3 encoder spec.")
else:
    print("\n→ Жодного робочого security-domain encoder локально не знайдено (або")
    print("  torch/transformers відсутні, або немає релевантних моделей у кеші, або")
    print("  жоден кандидат не пройшов sanity-check inference).")
    print("  Нічого не встановлено і не завантажено автоматично цим ноутбуком.")
    print("  Варіанти для обговорення: (а) завантажити конкретну security-domain модель")
    print("  свідомо, (б) почати з TF-IDF як контрольованого не-transformer baseline для B3.")
